# S&P 500 European Options Pricing and Model Validation

## Introduction

## Importing Libraries

In [2]:
import numpy as np
import pandas as pd
import yfinance as yf
import scipy as sp
import fractions as fr
import datetime as dt

## S&P 500 Historical Data

In [18]:
df_index = yf.download('^GSPC', period='2y')
df_close = df_index['Close']
S = df_close.iloc[-1,0]
q = 0.0105 #https://www.dividenddata.co.uk/sp500-dividend-yields.py
print(f'The most recent closing price from this data is {S}')
print(f'{q} is the continuous dividend yield, which has been estimated from the most recently published S&P 500 dividend yield. For the model, assume this stays constant.')

[*********************100%***********************]  1 of 1 completed

The most recent closing price from this data is 7729.72998046875
0.0105 is the continuous dividend yield, which has been estimated from the most recently published S&P 500 dividend yield. For the model, assume this stays constant.


## Estimating Volatility

In [19]:
df_returns = np.log(df_close/df_close.shift(1)).dropna()
sigma = (df_returns.std()).iloc[0] * np.sqrt(252)
print(f'{sigma} is our estimate of the annualised volatility.')


0.16059208426353336 is our estimate of the annualised volatility.


## User Inputs

In [24]:
allowed_type = ['call', 'Call', 'Put', 'put']
while True: #user enters call or put
    option_type = input('If the option is a call option, enter call. If the option is a put option, enter put.')
    if option_type in allowed_type:
        break
    else:
        print('Please enter either call or put.')

while True: #user enters maturity
    try:
        T = float(fr.Fraction(input('Enter the maturity time of the option in years.')))
    except ValueError:
        print('Please enter a number')
        continue
    if T>=1/12 and T<=30:
        break
    else:
        print('Please enter a number of years between 1/12 and 30.')

while True: #user enters strike price
    try:
        k = float(input('Enter the strike price'))
    except ValueError:
        print('Please enter a number')
        continue
    if k>0:
        break
    else:
        print('Please enter a positive number.')

## Approximating r

In [22]:
datenow = dt.date.today().year
treasury = f"https://home.treasury.gov/resource-center/data-chart-center/interest-rates/daily-treasury-rates.csv/{datenow}/all?type=daily_treasury_yield_curve&field_tdr_date_value={datenow}&page&_format=csv" #US daily treasury rates are low risk so base estimation off this
df_treasury = pd.read_csv(treasury).head(1)
print(df_treasury)
maturities = np.array([1/12, 1/8, 1/6, 1/4, 1/3, 1/2, 1, 2, 3, 5, 7, 10, 20, 30]) #in years
rates = (df_treasury.iloc[0,1:]).to_numpy(dtype=np.float64) / 100
r = np.interp(T, maturities, rates)
print(f'{r:.4f} is the approximated annualised risk free interest rate')

         Date  1 Mo  1.5 Month  2 Mo  3 Mo  4 Mo  6 Mo  1 Yr  2 Yr  3 Yr  \
0  10/27/2026   4.1       4.11  4.18  4.22  4.31  4.34   4.5   4.7   4.8   

   5 Yr  7 Yr  10 Yr  20 Yr  30 Yr  
0  4.82  4.91   4.99   5.34    5.3  
0.0450 is the approximated annualised risk free interest rate


## Calculating Theoretical Option Price

In [25]:
d_1 = (np.log(S/k) + T*(r - q + ((sigma)**2)/2))/(sigma*np.sqrt(T))
d_2 = d_1 - sigma*np.sqrt(T)
c_price = S*np.exp(-q*T)*sp.stats.norm.cdf(d_1) - k*np.exp(-r*T)*sp.stats.norm.cdf(d_2)
p_price = k*np.exp(-r*T)*sp.stats.norm.cdf(-d_2) - S*np.exp(-q*T)*sp.stats.norm.cdf(-d_1)

if option_type == 'call' or option_type == 'Call':
    print(f'{c_price:.2f} index points is the call option price.')
else:
    print(f'{p_price:.2f} index points is the put option price.')

print(f"S = {S}")
print(f"K = {k}")
print(f"T = {T}")
print(f"r = {r}")
print(f"q = {q}")
print(f"sigma = {sigma}")

362.68 index points is the put option price.
S = 7729.72998046875
K = 7730.0
T = 1.0
r = 0.045
q = 0.0105
sigma = 0.16059208426353336


## Testing Put-Call Parity

In [30]:
lhs = c_price - p_price
rhs = S*np.exp(-q*T)-k*np.exp(-r*T)
if np.isclose(lhs, rhs) == True:
    print('The parity equation has been satisfied')
else:
    print('The parity equation has not been satisfied')

The parity equation has been satisfied
